# Models training and tuning

Firstly, we import the training and testing datasets and configure the features

In [25]:
"""Import training and testing datasets"""
import pandas as pd

TRAINING_PATH = "../data/prepared/tennis_training.xlsx"
TESTING_PATH = "../data/prepared/tennis_testing.xlsx"
RAW_TESTING_PATH = "../data/testing/tennis_testing.xlsx"

train_set = pd.read_excel(TRAINING_PATH)
test_set = pd.read_excel(TESTING_PATH)

In [26]:
"""Features"""
target = "y"
debug_features = [
    "Date",
    "Player_1", "Player_2"
]
categorical_features = [
    "Series",
    "Court", 
    "Surface",
    "Round"
]

high_cardinality_categorical_features = [
    "Tournament",
]

numerical_features = [
    "Rank_Diff", "Points_Diff", "Best_of_5",
    "Odds_Diff",  # market log-odds ratio: log(odds_2 / odds_1)
    "Recent_Form_Diff", 
    "Dominance_Form_Diff",
    "Fatigue_Diff",
    "Recent_Surface_Form_Diff",
    "Elo_Diff", 
    "Surface_Elo_Diff",
    "H2H_Diff", "H2H_Surface_Diff",
    "Elo_Diff_x_EarlyRound",
    "Age_Diff"
]

features = numerical_features + categorical_features + high_cardinality_categorical_features

X_train, y_train = train_set[features], train_set[target]
X_test, y_test = test_set[features], test_set[target]

### Pipelines

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")), # fill missing values with most frequent value
    ("onehot", OneHotEncoder(handle_unknown="ignore")),   # use one-hot encoding
])

# high-cardinality features with one hot encoding, considering only k most frequent values
high_cardinality_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        max_categories=15,
    )),
])

### Logistic Regression

We use a standard scaler, as logistic regression is sensitive to the scale of the features.

In [28]:
logreg_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True,
    )),
    ("scaler", StandardScaler()),
])

logreg_preprocessor = ColumnTransformer([
    ("num", logreg_numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features),
    ("high_cardinality", high_cardinality_pipeline, high_cardinality_categorical_features),
])

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import loguniform, uniform


logreg_pipeline = Pipeline([
    ("preprocessor", logreg_preprocessor),
    ("model", LogisticRegression(
        solver="saga", # works with l1 and elasticnet penalties
        max_iter=5000, # saga converges slowly
        random_state=42,
    )),
])

param_distributions = {
    "model__C": loguniform(1e-3, 100),
    "model__l1_ratio": uniform(0, 1),  # 0 = L2, 1 = L1, intermediate values = Elastic Net
}

time_cv = TimeSeriesSplit(n_splits=5)

logreg_search = RandomizedSearchCV(
    estimator=logreg_pipeline,
    param_distributions=param_distributions,
    n_iter=150,
    scoring="neg_log_loss",
    cv=time_cv,
    random_state=42,
    verbose=1,
    refit=True,
    n_jobs=-1,
)

logreg_search.fit(X_train, y_train)

print("Best params:", logreg_search.best_params_)
print("CV log loss:", f"{-logreg_search.best_score_:.4f}")

Fitting 5 folds for each of 150 candidates, totalling 750 fits
Best params: {'model__C': np.float64(0.01967432802530612), 'model__l1_ratio': np.float64(0.662522284353982)}
CV log loss: 0.5702


### XGBoost

#### Preprocessor

The preprocessor involves cleaning, transforming and normalizing the data to make it suitable for model training. This includes handling missing values, encoding categorical variables and scaling numerical features.

In [30]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",          # fill missing values with median
        add_indicator=True,         # add indicator column for missing values
    )),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features),
    ("high_cardinality", high_cardinality_pipeline, high_cardinality_categorical_features),
])

#### Randomized CV

We use `TimeSeriesSplit` to avoid data leakage, as we are dealing with time series data.

In [31]:
from xgboost import XGBClassifier
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

xgb_search_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )),
])

param_distributions = {
    "model__n_estimators": randint(150, 800),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__max_depth": randint(2, 8),
    "model__min_child_weight": loguniform(0.5, 10),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__reg_alpha": loguniform(1e-4, 10),
    "model__reg_lambda": loguniform(0.1, 20),
}

time_cv = TimeSeriesSplit(n_splits=5)

xgb_search = RandomizedSearchCV(
    estimator=xgb_search_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="neg_log_loss",
    cv=time_cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

xgb_search.fit(X_train, y_train)

print("Best params:", xgb_search.best_params_)
print("CV log loss:", f"{-xgb_search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'model__colsample_bytree': np.float64(0.9530545372757359), 'model__learning_rate': np.float64(0.017600038115609038), 'model__max_depth': 2, 'model__min_child_weight': np.float64(1.423318635740201), 'model__n_estimators': 357, 'model__reg_alpha': np.float64(3.0588015371390562), 'model__reg_lambda': np.float64(10.995436403641982), 'model__subsample': np.float64(0.9119502183430496)}
CV log loss: 0.5712


### Saving the models

In [32]:
import pickle
from pathlib import Path

logistic_regression_model_path = Path("../models/logistic_regression_model.pkl")
xgboost_model_path = Path("../models/xgboost_model.pkl")

logistic_regression_model_path.parent.mkdir(parents=True, exist_ok=True)
xgboost_model_path.parent.mkdir(parents=True, exist_ok=True)

with logistic_regression_model_path.open("wb") as file:
    pickle.dump(logreg_search.best_estimator_, file)

with xgboost_model_path.open("wb") as file:
    pickle.dump(xgb_search.best_estimator_, file)

print(f"Saved Logistic Regression model to {logistic_regression_model_path}")
print(f"Saved XGBoost model to {xgboost_model_path}")

Saved Logistic Regression model to ../models/logistic_regression_model.pkl
Saved XGBoost model to ../models/xgboost_model.pkl
